<a href="https://colab.research.google.com/github/haqiaryva/UTS-DeepLearning/blob/main/Studi_Kasus_1_MLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.neural_network import MLPClassifier

In [3]:
train_data = pd.read_csv('train.csv')
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
train_data.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [5]:
train_data["Age"] = train_data["Age"].fillna(train_data["Age"].median())
train_data["Embarked"] = train_data["Embarked"].fillna(train_data["Embarked"].mode()[0])
train_data.drop(columns=["Cabin"], inplace=True)
train_data.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,0


In [6]:
train_data["Sex"] = train_data["Sex"].map({"male": 0, "female": 1})
train_data["Embarked"] = train_data["Embarked"].map({"S": 0, "C": 1, "Q": 2})
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,A/5 21171,7.2500,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,1
2,3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,STON/O2. 3101282,7.9250,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,0
4,5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,373450,8.0500,0


In [7]:
train_data["FamilySize"] = train_data["SibSp"] + train_data["Parch"] + 1
train_data["IsAlone"] = (train_data["FamilySize"] == 1).astype(int)
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,FamilySize,IsAlone
0,1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,A/5 21171,7.2500,0,2,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,1,2,0
2,3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,STON/O2. 3101282,7.9250,0,1,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,0,2,0
4,5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,373450,8.0500,0,1,1


In [8]:
binn = [0,12,25,60,100]
label = ["anak-anak", "remaja", "dewasa", "lansia"]
train_data["AgeGroup"] = pd.cut(train_data["Age"], bins=binn, labels=label)

le = LabelEncoder()
train_data["AgeGroup"] = le.fit_transform(train_data["AgeGroup"])

train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,FamilySize,IsAlone,AgeGroup
0,1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,A/5 21171,7.2500,0,2,0,3
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,1,2,0,1
2,3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,STON/O2. 3101282,7.9250,0,1,1,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,0,2,0,1
4,5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,373450,8.0500,0,1,1,1


In [9]:
train_data["Title"] = train_data["Name"].str.extract(r' ([A-Za-z]+)\.', expand=False)

title_counts = train_data["Title"].value_counts()
print("Sebelum cleaning:\n", title_counts)

Sebelum cleaning:
 Title
Mr          517
Miss        182
Mrs         125
Master       40
Dr            7
Rev           6
Col           2
Mlle          2
Major         2
Ms            1
Mme           1
Don           1
Lady          1
Sir           1
Capt          1
Countess      1
Jonkheer      1
Name: count, dtype: int64


In [10]:
rare_titles = ["Lady", "Countess", "Capt", "Col", "Don",
                   "Dr", "Major", "Rev", "Sir", "Jonkheer", "Dona"]
train_data["Title"] = train_data["Title"].replace(rare_titles, "Rare")
train_data["Title"] = train_data["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
train_data["Title"] = le.fit_transform(train_data["Title"])

train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,FamilySize,IsAlone,AgeGroup,Title
0,1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,A/5 21171,7.2500,0,2,0,3,2
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,1,2,0,1,3
2,3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,STON/O2. 3101282,7.9250,0,1,1,1,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,0,2,0,1,3
4,5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,373450,8.0500,0,1,1,1,2


In [11]:
train_data= train_data.drop(columns=["Name", "Ticket", "PassengerId"])
train_data.head()


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,FamilySize,IsAlone,AgeGroup,Title
0,0,3,0,22.0,1,0,7.2500,0,2,0,3,2
1,1,1,1,38.0,1,0,71.2833,1,2,0,1,3
2,1,3,1,26.0,0,0,7.9250,0,1,1,1,1
3,1,1,1,35.0,1,0,53.1000,0,2,0,1,3
4,0,3,0,35.0,0,0,8.0500,0,1,1,1,2


In [12]:
X = train_data.drop(columns=['Survived'])
y = train_data['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [13]:
mlp = MLPClassifier(
    hidden_layer_sizes=(24),
    max_iter=1000,
    activation='relu',
    solver='adam',
    random_state=42,
    batch_size=8)
mlp.fit(X_train_scaled, y_train)

print(f"Training Accuracy: {mlp.score(X_train_scaled, y_train):.2f}")
print(f"Testing Accuracy: {mlp.score(X_test_scaled, y_test):.2f}")

Training Accuracy: 0.87
Testing Accuracy: 0.82


In [14]:
y_pred = mlp.predict(X_test_scaled)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.82      0.89      0.85       105
           1       0.82      0.73      0.77        74

    accuracy                           0.82       179
   macro avg       0.82      0.81      0.81       179
weighted avg       0.82      0.82      0.82       179



In [15]:
print("--- Arsitektur Model MLP ---")
print(f"Fitur Input: {mlp.n_features_in_}")
print(f"Hidden Layers: {mlp.hidden_layer_sizes}")
print(f"Output Classes: {len(mlp.classes_)}")
print(f"Fungsi Aktivasi: {mlp.activation}")

--- Arsitektur Model MLP ---
Fitur Input: 11
Hidden Layers: 24
Output Classes: 2
Fungsi Aktivasi: relu


In [16]:
test_data = pd.read_csv("test.csv")
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [17]:
passenger_ids = test_data["PassengerId"]

# Drop same columns as training
# 1. Ekstrak Title (masih punya kolom Name)
test_data["Title"] = test_data["Name"].str.extract(r' ([A-Za-z]+)\.', expand=False)

# 2. Bersihkan Title (rare titles, dll.)
rare_titles = ["Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major", "Rev", "Sir", "Jonkheer", "Dona"]
test_data["Title"] = test_data["Title"].replace(rare_titles, "Rare")
test_data["Title"] = test_data["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})

# 3. Encode Title (gunakan encoder yang sudah di-fit dari training jika ada)
test_data["Title"] = le.fit_transform(test_data["Title"]) # le_title dari training

# 4. Sekarang baru drop kolom yang tidak perlu (termasuk Name)
test_data = test_data.drop(columns=["Name", "Ticket", "PassengerId", "Cabin"])

# 5. Fill missing values
test_data["Age"] = test_data["Age"].fillna(test_data["Age"].median())
test_data["Embarked"] = test_data["Embarked"].fillna(test_data["Embarked"].mode()[0])
test_data["Fare"] = test_data["Fare"].fillna(test_data["Fare"].median())

# 6. Encode categorical variables
test_data["Sex"] = test_data["Sex"].map({"male": 0, "female": 1})
test_data["Embarked"] = test_data["Embarked"].map({"S": 0, "C": 1, "Q": 2})

test_data["FamilySize"] = test_data["SibSp"] + train_data["Parch"] + 1
test_data["IsAlone"] = (test_data["FamilySize"] == 1).astype(int)

binn = [0,12,25,60,100]
label = ["anak-anak", "remaja", "dewasa", "lansia"]
test_data["AgeGroup"] = pd.cut(test_data["Age"], bins=binn, labels=label)
test_data["AgeGroup"] = le.fit_transform(test_data["AgeGroup"])

# 7. Align columns with training data
test_data = test_data[X_train.columns]

# 8. Make predictions
predictions = mlp.predict(test_data)

# Create submission file
submission = pd.DataFrame({
    "PassengerId": passenger_ids,
    "Survived": predictions
})
submission.to_csv('MLP.csv', index=False)


submission.head()

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MLPClassifier was fitted without feature names
  warnings.warn(


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,1
4,896,1


In [18]:
MLP = pd.read_csv('MLP.csv')
survival_counts = MLP['Survived'].value_counts()
display(survival_counts)

,count
Survived,
1,319
0,99
